Pandas in Spark

In [2]:
from pyspark.sql import SparkSession
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

import pyspark.pandas as ps
# from pyspark.pandas import pandas as ps



spark = SparkSession.builder \
    .appName('Pandas on Spark') \
    .config('spark.sql.ansi.enable', 'False')\
    .config('spark.executerEnv.PYARROW_IGNORE_TIMEZONE', '1')\
    .getOrCreate()

# Setting log levels in pyspark scripts
spark.sparkContext.setLogLevel('WARN')

pandasSpark_df = ps.DataFrame({
    'id' : [1, 2, 3, 4, 5],
    'name' : ['Alice', 'Bob', 'Charlie', 'David', 'Emma'],
    'age' : [25, 30, 35, 40, 45],
    'salary' : [50000, 60000, 75000, 80000, 120000]
})

print(pandasSpark_df)
print('\nAvg Age: ', pandasSpark_df['age'].mean())
print('\n Summary Statistics: ', pandasSpark_df.describe())

pandasSpark_df['salary_after_promotion'] = pandasSpark_df['salary']* 1.1
print('\n DataFrame with Promoted Salaries:', pandasSpark_df)

filtered_pandasSpark_df = pandasSpark_df[pandasSpark_df['age'] > 30]
print('\n Filtered DataFrame (Age > 30):', filtered_pandasSpark_df)

spark_df = pandasSpark_df.to_spark()
spark_df.show()

ps_df_from_spark = ps.DataFrame(spark_df)
print('\nReconverted pandas-on-spark dataframe: ', ps_df_from_spark)



spark.stop()


    
    
    



   id     name  age  salary
0   1    Alice   25   50000
1   2      Bob   30   60000
2   3  Charlie   35   75000
3   4    David   40   80000
4   5     Emma   45  120000

Avg Age:  35.0

 Summary Statistics:  

             id        age        salary
count  5.000000   5.000000       5.00000
mean   3.000000  35.000000   77000.00000
std    1.581139   7.905694   26832.81573
min    1.000000  25.000000   50000.00000
25%    2.000000  30.000000   60000.00000
50%    3.000000  35.000000   75000.00000
75%    4.000000  40.000000   80000.00000
max    5.000000  45.000000  120000.00000

 DataFrame with Promoted Salaries:    id     name  age  salary  salary_after_promotion
0   1    Alice   25   50000                 55000.0
1   2      Bob   30   60000                 66000.0
2   3  Charlie   35   75000                 82500.0
3   4    David   40   80000                 88000.0
4   5     Emma   45  120000                132000.0

 Filtered DataFrame (Age > 30):    id     name  age  salary  salary_after_promotion
2   3  Charlie   35   75000                 82500.0
3   4    David   40   80000                 88000.0
4   5     Emma   45  120000                132000.0


/home/carter/RevProjects/Spark/.venv/lib/python3.14/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


+---+-------+---+------+----------------------+
| id|   name|age|salary|salary_after_promotion|
+---+-------+---+------+----------------------+
|  1|  Alice| 25| 50000|     55000.00000000001|
|  2|    Bob| 30| 60000|               66000.0|
|  3|Charlie| 35| 75000|               82500.0|
|  4|  David| 40| 80000|               88000.0|
|  5|   Emma| 45|120000|              132000.0|
+---+-------+---+------+----------------------+


Reconverted pandas-on-spark dataframe:     id     name  age  salary  salary_after_promotion
0   1    Alice   25   50000                 55000.0
1   2      Bob   30   60000                 66000.0
2   3  Charlie   35   75000                 82500.0
3   4    David   40   80000                 88000.0
4   5     Emma   45  120000                132000.0


Same code, just different things than above

In [ ]:
from pyspark.sql import SparkSession
import os
os.environ['PYARROW_IGNORE_TIMEZONE'] = '1'

import pyspark.pandas as ps
# from pyspark.pandas import pandas as ps



spark = SparkSession.builder \
    .appName('Pandas on Spark') \
    .config('spark.sql.ansi.enable', 'False')\
    .config('spark.executerEnv.PYARROW_IGNORE_TIMEZONE', '1')\
    .getOrCreate()

# Setting log levels in pyspark scripts
spark.sparkContext.setLogLevel('WARN')

ps_df = ps.DataFrame({
    'id' : [1, 2, 3, 4, 5],
    'name' : ['Alice', 'Bob', 'Charlie', 'David', 'Emma'],
    'age' : [25, 30, 35, 40, 45],
    'salary' : [50000, 60000, 75000, 80000, 120000]
})
# Using transform for element-wise operations
ps_df['age_in_10_years'] = ps_df['age'].transform(lambda x: x + 10)
print(ps_df)

# using apply(); func that will run for each value in a column

def categorize_salary(salary):
    if salary < 40000:
        return 'Low'
    elif salary < 80000:
        return 'Medium'
    else:
        return 'High'

ps_df['Salary Category'] = ps_df['salary'].apply(categorize_salary)
print(ps_df)

# Apply function accross rows

def format_row(row):
    return f"{row['name']}: ({row['age']} years old)"

# With the (axis=1) part, this acts as changing to side-to-side form vertical

ps_df['Name_With_Age'] = ps_df.apply(format_row, axis=1)
print(ps_df)
spark.stop()


    
    
    



   id     name  age  salary  age_in_10_years
0   1    Alice   25   50000               35
1   2      Bob   30   60000               40
2   3  Charlie   35   75000               45
3   4    David   40   80000               50
4   5     Emma   45  120000               55
   id     name  age  salary  age_in_10_years Salary Category
0   1    Alice   25   50000               35          Medium
1   2      Bob   30   60000               40          Medium
2   3  Charlie   35   75000               45          Medium
3   4    David   40   80000               50            High
4   5     Emma   45  120000               55            High


/home/carter/RevProjects/Spark/.venv/lib/python3.14/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


   id     name  age  salary  age_in_10_years Salary Category           Name_With_Age
0   1    Alice   25   50000               35          Medium    Alice: (25 years old
1   2      Bob   30   60000               40          Medium      Bob: (30 years old
2   3  Charlie   35   75000               45          Medium  Charlie: (35 years old
3   4    David   40   80000               50            High    David: (40 years old
4   5     Emma   45  120000               55            High     Emma: (45 years old
